# 04 - QLoRA Fine-Tuning Diagnostic (v4)

Thin Kaggle runner for the `v4_diagnostic` smoke run. It verifies the v4 training stack (diverse output formats, expanded implications) before the full `v4_improved` run.

Run full training only after this diagnostic notebook passes and artifacts are inspected.

In [ ]:
!pip install -q -U datasets transformers trl peft accelerate bitsandbytes pyyaml
!pip install -q -U unsloth_zoo
!pip install -q -U --no-deps unsloth

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/W-Kaski/fingpt-qlora.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "fingpt-qlora"
CONFIG = "configs/experiments/v4_improved.yaml"

os.chdir(WORKDIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir(REPO_DIR)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Repo commit: {commit}")

In [ ]:
# Validate local control-plane code before spending GPU time.
!python -m unittest discover tests
!python -m compileall -q src tests


In [ ]:
# Build or refresh data splits.
!python -m src.data.download
!python -m src.data.preprocess
!python -m src.data.format_chat
!python -m src.data.merge_datasets
!python -m src.data.splits
!python -m src.data.manifest --splits-dir data/splits --output results/data_manifest_v3.json


In [ ]:
import json

with open("data/splits/sharegpt_all.json") as f:
    all_data = json.load(f)

# Check output format diversity (v4 has 5 variants)
samples = [ex["conversations"][-1]["value"] for ex in all_data[:100]]
has_sentiment_header = sum(1 for s in samples if "## Sentiment Analysis" in s)
has_bold_label = sum(1 for s in samples if "**Positive**" in s or "**Negative**" in s or "**Neutral**" in s)
has_driver = sum(1 for s in samples if "The primary driver is" in s)
has_concise = sum(1 for s in samples if s.startswith("**"))

print(f"Sample of {len(samples)} outputs:")
print(f"  Classic header (## Sentiment Analysis): {has_sentiment_header}")
print(f"  Bold label variant: {has_bold_label}")
print(f"  Driver-first variant: {has_driver}")
print(f"  Concise variant: {has_concise}")
print(f"\nFirst 3 samples:")
for i, s in enumerate(samples[:3]):
    print(f"  [{i}] {s[:120]}...")

In [ ]:
# Train LoRA adapter. All hyperparameters come from CONFIG.
!python -m src.train.train_sft --config {CONFIG} --output-suffix kaggle

In [ ]:
from pathlib import Path

ADAPTER_DIR = "outputs/v4_improved-kaggle"

for artifact in [
    "results/data_manifest_v3.json",
    f"{ADAPTER_DIR}/run_manifest.json",
    f"{ADAPTER_DIR}/trainer_state.json",
    f"{ADAPTER_DIR}/config_used.yaml",
]:
    path = Path(artifact)
    print(f"{artifact}: {path.exists()}")
    assert path.exists(), artifact

print("\nAll artifacts present.")

## Quick Test Evaluation (5 samples)

Load the trained adapter, run inference on 5 test samples, verify format compliance and sentiment accuracy before full evaluation.

In [ ]:
import json
import gc
import torch
from unsloth import FastLanguageModel
from src.eval.benchmark import build_prompt
from src.eval.run_evaluation import extract_sentiment_label, format_compliance

ADAPTER_DIR = "outputs/v4_improved-kaggle"
N_SAMPLES = 5

# Load test data
with open("data/splits/test.json") as f:
    test_data = json.load(f)

# Pick first N samples
test_subset = test_data[:N_SAMPLES]

# Load adapter
print(f"Loading adapter from {ADAPTER_DIR}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# Run inference
results = []
for i, example in enumerate(test_subset):
    messages = build_prompt(example["conversations"])
    reference = example["conversations"][-1]["value"]

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(input_ids=input_ids, max_new_tokens=256, do_sample=False)
    prediction = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)

    ref_label = extract_sentiment_label(reference)
    pred_label = extract_sentiment_label(prediction)

    results.append({
        "reference": reference,
        "prediction": prediction,
        "ref_label": ref_label,
        "pred_label": pred_label,
        "correct": ref_label == pred_label,
    })

    print(f"\n--- Sample {i} ---")
    print(f"REF:  {reference[:120]}...")
    print(f"PRED: {prediction[:120]}...")
    print(f"Label: {ref_label} -> {pred_label} {'OK' if ref_label == pred_label else 'WRONG'}")

del model
torch.cuda.empty_cache()
gc.collect()

# Save for next cell
with open("/tmp/quick_eval_results.json", "w") as f:
    json.dump(results, f)

In [ ]:
import json
import re

with open("/tmp/quick_eval_results.json") as f:
    results = json.load(f)

N = len(results)
correct = sum(1 for r in results if r["correct"])
predictions = [r["prediction"] for r in results]

# Format compliance (new v4 regex)
label_pattern = r"\*\*(?:Sentiment:\s*)?(?:Positive|Negative|Neutral)\*\*"
compliant = sum(1 for p in predictions if re.search(label_pattern, p, re.IGNORECASE))

# Output diversity
has_classic = sum(1 for p in predictions if "## Sentiment Analysis" in p)
has_bold = sum(1 for p in predictions if "**Positive**" in p or "**Negative**" in p or "**Neutral**" in p)

print("=" * 50)
print(f"QUICK EVAL SUMMARY ({N} samples)")
print("=" * 50)
print(f"  Sentiment accuracy:  {correct}/{N} = {correct/N:.0%}")
print(f"  Format compliance:   {compliant}/{N} = {compliant/N:.0%}")
print(f"  Classic header:      {has_classic}/{N}")
print(f"  Bold label variant:  {has_bold}/{N}")
print()

# Full prediction details
for i, r in enumerate(results):
    status = "OK" if r["correct"] else "WRONG"
    print(f"  [{i}] {r['ref_label']:8s} -> {r['pred_label']:8s}  {status}")
    print(f"       {r['prediction'][:150]}")
    print()

if correct >= N * 0.6 and compliant >= N * 0.8:
    print("PASS - Ready for full evaluation")
else:
    print("REVIEW NEEDED - Check predictions above")